# VitroVision รอบ 2 — SAM3 วิเคราะห์ 100 ขวด พริกจินดา
รัน: 17 ส.ค. 2569 · ชุดภาพ 20260814_batch (001–100.jpg) · species = พริกจินดา (Capsicum frutescens)
> ก่อนรัน: ตั้ง Colab secret ชื่อ `HF_TOKEN` (หน้า 🔑 Secrets) หรือวาง token ใน cell ที่ 3

In [ ]:
# 1) ติดตั้ง dependency
!pip install -q transformers torch torchvision opencv-python pillow matplotlib pandas numpy huggingface_hub xlsxwriter openpyxl
print("deps OK")

In [ ]:
# 2) Login Hugging Face — ต้องมี access ถึง facebook/sam3 (gated)
from huggingface_hub import login
import os
token = ""
try:
    from google.colab import userdata
    token = userdata.get("HF_TOKEN", "")
except Exception:
    pass
if not token:
    token = os.environ.get("HF_TOKEN", "")
if not token:
    token = input("วาง HF_TOKEN แล้วกด Enter: ").strip()
login(token=token, add_to_git_credential=False)
print("HF login:", "OK" if token else "FAILED")

In [ ]:
# 3) Mount Google Drive (เก็บภาพ + ผลลัพธ์)
from google.colab import drive
drive.mount("/content/drive")
print("Drive mounted")

In [ ]:
# 4) เตรียมข้อมูลจาก Drive (โฟลเดอร์ vitrovision_round2 ใน MyDrive)
import os, zipfile, shutil
DRIVE_BASE = "/content/drive/MyDrive/vitrovision_round2"
DATA_DIR = "/content/data"; OUT_DIR = "/content/results"
os.makedirs(DATA_DIR, exist_ok=True); os.makedirs(OUT_DIR, exist_ok=True)

shutil.copy(f"{DRIVE_BASE}/sam3_growth_pipeline.py", "/content/sam3_growth_pipeline.py")
with zipfile.ZipFile(f"{DRIVE_BASE}/20260814_batch.zip") as z:
    z.extractall(DATA_DIR)
for extra in ("species_map.csv", "manifest.csv"):
    p = f"{DRIVE_BASE}/{extra}"
    if os.path.exists(p):
        shutil.copy(p, DATA_DIR)
imgs = sorted(f for f in os.listdir(DATA_DIR) if f.lower().endswith(".jpg"))
print(f"ภาพพร้อม: {len(imgs)} ไฟล์ (ตัวอย่าง: {imgs[:3]} ... {imgs[-1]})")

In [ ]:
# 5) รัน pipeline (SAM3 5 prompts + ROI ขวด + verdict 3 คลาส + checkpoint)
import time
t0 = time.time()
!python /content/sam3_growth_pipeline.py --data /content/data --out /content/results
print(f"รันเสร็จใน {(time.time()-t0)/60:.1f} นาที")

In [ ]:
# 6) บันทึกผลลง Drive + ดาวน์โหลดกลับเครื่อง
import shutil, os
shutil.make_archive("/content/results_round2", "zip", "/content/results")
shutil.copy("/content/results_round2.zip", f"{DRIVE_BASE}/results_round2_20260817.zip")
print("บันทึก Drive:", f"{DRIVE_BASE}/results_round2_20260817.zip")
from google.colab import files
files.download("/content/results_round2.zip")